# 06 · Emission integration, F2, the grouped two-channel emission

Runs the F2 grid, joint-trained BKT whose emission carries the two grouped misconception chains, conceptual and procedural, as separate additive channels beside slip, every capture rate a named parameter. No pooling anywhere, the additive sum is the combination rule, so each group keeps its own beta all the way into the emission and the central readout is the conceptual against procedural split.

The chains are GroupedChains under the chain of record, strong snap, fitted once and shared frozen across every row. The channel inputs at turn k are the two filtered states before that turn, solution row consumed by the chains, never by BKT. Evaluation is paper-aligned, each dialogue's first scored turn updates the filter and is excluded from metrics, unseen KCs contribute 0.5, exactly 0.5 classifies to 0.

**The rows:**

- Validity, beta pinned to 0, the engine's like-for-like BKT refit. Expected to land within a few tenths of the frozen M1, 60.65 accuracy, 64.28 AUC, 55.60 f1, and to match notebook 05's validity row closely, the residual being optimizer and initialization, not protocol.
- The grid, three connections, mastered, unmastered, both, each under free betas and betas pinned to 1.

**Reading order.** Validity first, then the free mastered row's two betas, the split is the finding this rung exists for, then the deltas in section 4. Fitted betas read as lower bounds, attenuated by chain measurement noise. On the both rows the four betas fit independently and the mastered against unmastered split locates where capture acts.

## 1. Setup

Grouped chains fitted once, shared by every row.

In [1]:
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.chain import TriggerChain
from scripts.misconception_chains_grouped import GroupedChains
from scripts.emission_integration_pooled import CHAIN_OF_RECORD
from scripts.emission_integration_grouped import (
    EmissionIntegrationGroupedMastered,
    EmissionIntegrationGroupedUnmastered,
    EmissionIntegrationGroupedBoth,
)

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

chains = GroupedChains(train_df, test_df, chain_class=TriggerChain,
                       chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
chains.summary()

,chain,pi,onset,resolve,pP_i,pP_a,expected_dwell_turns,accuracy,tpr,tnr,auc,f1,informative_cells
0,conceptual,0.05,0.0,0.0348,0.02,0.97,28.7,0.6324,0.6683,0.4100,0.5801,0.7579,1439
1,procedural,0.05,0.0,0.0870,0.02,0.97,11.5,0.6739,0.5610,0.8047,0.7250,0.6487,1282


## 2. Rows

Seven fits, loop-built so each row's configuration is its name.

In [2]:
CONNECTIONS = {
    "mastered": EmissionIntegrationGroupedMastered,
    "unmastered": EmissionIntegrationGroupedUnmastered,
    "both": EmissionIntegrationGroupedBoth,
}

ROWS = {"validity, beta=0": (EmissionIntegrationGroupedMastered,
                             {"pin_beta": 0.0})}
for connection, cls in CONNECTIONS.items():
    ROWS[f"{connection}, free"] = (cls, {})
    ROWS[f"{connection}, beta=1"] = (cls, {"pin_beta": 1.0})

models = {}
for name, (cls, kwargs) in ROWS.items():
    model = cls(train_df, test_df, chains=chains, **kwargs)
    model.run()
    models[name] = model
    print(f"{name:22s} {model.metrics}")

validity, beta=0       {'accuracy': 0.6035, 'auc': 0.6397, 'f1': 0.5531, 'turns': 1985, 'beta_mastered_conceptual': 0.0, 'beta_mastered_procedural': 0.0, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
mastered, free         {'accuracy': 0.603, 'auc': 0.6374, 'f1': 0.5429, 'turns': 1985, 'beta_mastered_conceptual': 0.1849, 'beta_mastered_procedural': 0.1115, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
mastered, beta=1       {'accuracy': 0.605, 'auc': 0.6417, 'f1': 0.5767, 'turns': 1985, 'beta_mastered_conceptual': 1.0, 'beta_mastered_procedural': 1.0, 'beta_unmastered_conceptual': None, 'beta_unmastered_procedural': None, 'connection': 'mastered'}
unmastered, free       {'accuracy': 0.6045, 'auc': 0.6361, 'f1': 0.5754, 'turns': 1985, 'beta_mastered_conceptual': None, 'beta_mastered_procedural': None, 'beta_unmastered_conceptual': 0.302, 'beta_unmastered_procedural': 0.2435, 'connecti

## 3. Results table

The frozen M1 trio is the anchor, deltas against it are context, section 4 carries the inference.

In [3]:
M1 = {"accuracy": 0.6065, "auc": 0.6428, "f1": 0.5560}

results = pd.DataFrame(
    [{"row": name, **models[name].metrics} for name in ROWS])
for metric in ("accuracy", "auc", "f1"):
    results[f"d_{metric}"] = (results[metric] - M1[metric]).round(4)
results = results.set_index("row")
results[["connection", "beta_mastered_conceptual", "beta_mastered_procedural",
         "beta_unmastered_conceptual", "beta_unmastered_procedural",
         "accuracy", "d_accuracy", "auc", "d_auc", "f1", "d_f1", "turns"]]

,connection,beta_mastered_conceptual,beta_mastered_procedural,beta_unmastered_conceptual,beta_unmastered_procedural,accuracy,d_accuracy,auc,d_auc,f1,d_f1,turns
row,,,,,,,,,,,,
"validity, beta=0",mastered,0.0000,0.0000,NaN,NaN,0.6035,-0.0030,0.6397,-0.0031,0.5531,-0.0029,1985
"mastered, free",mastered,0.1849,0.1115,NaN,NaN,0.6030,-0.0035,0.6374,-0.0054,0.5429,-0.0131,1985
"mastered, beta=1",mastered,1.0000,1.0000,NaN,NaN,0.6050,-0.0015,0.6417,-0.0011,0.5767,0.0207,1985
"unmastered, free",unmastered,NaN,NaN,0.3020,0.2435,0.6045,-0.0020,0.6361,-0.0067,0.5754,0.0194,1985
"unmastered, beta=1",unmastered,NaN,NaN,1.0000,1.0000,0.6020,-0.0045,0.6344,-0.0084,0.5789,0.0229,1985
"both, free",both,0.0660,0.0002,0.2342,0.2000,0.6060,-0.0005,0.6334,-0.0094,0.5670,0.0110,1985
"both, beta=1",both,1.0000,1.0000,1.0000,1.0000,0.5481,-0.0584,0.5943,-0.0485,0.2666,-0.2894,1985


## 4. Deltas against the engine baseline

Every model against the validity row's own three metrics, code path, protocol, and optimizer held fixed, so each delta isolates what the channel configuration changed. Sorted by AUC delta.

In [4]:
baseline = models["validity, beta=0"].metrics
deltas = pd.DataFrame([
    {"row": name,
     "d_accuracy": round(models[name].metrics["accuracy"]
                         - baseline["accuracy"], 4),
     "d_auc": round(models[name].metrics["auc"] - baseline["auc"], 4),
     "d_f1": round(models[name].metrics["f1"] - baseline["f1"], 4)}
    for name in ROWS if name != "validity, beta=0"])
deltas.set_index("row").sort_values("d_auc", ascending=False)

,d_accuracy,d_auc,d_f1
row,,,
"mastered, beta=1",0.0015,0.0020,0.0236
"mastered, free",-0.0005,-0.0023,-0.0102
"unmastered, free",0.0010,-0.0036,0.0223
"unmastered, beta=1",-0.0015,-0.0053,0.0258
"both, free",0.0025,-0.0063,0.0139
"both, beta=1",-0.0554,-0.0454,-0.2865


## 5. Slip attribution

Each KC's fitted slip on the free mastered row against the validity row's, same optimizer both sides, so the drop is the channel's re-attribution of what the baseline filed as unexplained slip.

In [5]:
m2 = models["mastered, free"]
bkt = models["validity, beta=0"]
slips = pd.DataFrame([
    {"kc": kc, "m2_slip": round(p["slip"], 4),
     "bkt_slip": round(bkt.parameters[kc]["slip"], 4)}
    for kc, p in m2.parameters.items()])
slips["drop"] = (slips["bkt_slip"] - slips["m2_slip"]).round(4)
print(f"median slip drop {slips['drop'].median():+.4f} over {len(slips)} KCs, "
      f"beta_mastered_conceptual {m2.metrics['beta_mastered_conceptual']}, "
      f"beta_mastered_procedural {m2.metrics['beta_mastered_procedural']}")
slips.sort_values("drop", ascending=False).head(10)

median slip drop +0.0027 over 138 KCs, beta_mastered_conceptual 0.1849, beta_mastered_procedural 0.1115


,kc,m2_slip,bkt_slip,drop
1,"Add and subtract within 1000, using concrete m...",0.0001,0.5531,0.5530
19,Compare two fractions with different numerator...,0.6915,0.9999,0.3084
4,Analyze and solve pairs of simultaneous linear...,0.3203,0.6000,0.2797
97,Solve systems of linear equations exactly and ...,0.3203,0.6000,0.2797
23,"Count to 120, starting at any number less than...",0.1628,0.4301,0.2673
65,Interpret multiplication as scaling (resizing)...,0.2504,0.5000,0.2496
30,Determine the unknown whole number in a multip...,0.1768,0.4152,0.2384
51,Find whole-number quotients of whole numbers w...,0.5695,0.7850,0.2155
114,Understand that the probability of a chance ev...,0.0826,0.2624,0.1798
69,Make a line plot to display a data set of meas...,0.8213,0.9999,0.1786


## 6. Interpretation ledger

- The validity row's distance from M1, and from notebook 05's validity row, bounds what initialization and the optimizer contribute, read every delta net of it.
- The free mastered row's two betas are the rung's central readout, whether conceptual belief captures at a different rate than procedural, each a lower bound under chain measurement noise. Compare their sum against notebook 05's single pooled beta, the scalar rate should sit near a weighted blend of the two.
- On the both rows the mastered and unmastered betas fit independently, a stable mastered pair beside drifting or bound-hitting unmastered values reads as competence-side capture, the unmastered branch's clipped likelihood absorbing variance rather than measuring.
- The pinned both row subtracts up to 2 from each branch, deep truncation, either group live asserts near-certain incorrectness, expect its f1 behavior to be the F1 pinned rows amplified.
- Grouping earns its parameters only if the free rows here beat notebook 05's corresponding free rows, that cross-notebook comparison is legitimate, identical universe, protocol, and engine, only the channel grain differs.

## 7. Why the grouped channels did not improve prediction

The two-channel grid inherits the F1 null and sharpens it, the free mastered row lands 0.2 AUC points below the engine baseline, so the second channel bought variance, not signal, and the run's information is in the fitted values rather than the metric deltas.

**The split is the rung's finding.** Conceptual captures harder than procedural, beta_mastered_conceptual 0.185 against beta_mastered_procedural 0.112, both small, both lower bounds under chain measurement noise, and their magnitudes are consistent with notebook 05's single pooled beta of 0.178, max pooling tracks the strongest chain and the conceptual group usually is the strongest, so the scalar was already mostly conceptual. The mechanism behind the smallness is unchanged from F1, the filtered states are persistence-wide, high across locally-correct turns inside live threads, and joint training lets mastery absorb what correctness history already implies, the betas are paid only for the residual.

**The both connection exposes credit migration.** With both branches open, the mastered betas collapse toward zero, 0.066 and 0.0002, while the unmastered pair absorbs 0.23 and 0.20, the two branches compete for one weak signal and EM routes it through the guess branch's clipped, poorly-identified likelihood, a warning against reading any both-row beta as a capture rate.

**The pinned rows split into a replication and a demonstration.** Mastered beta-equals-one replicates F1's calibration effect, AUC held, f1 up 0.024, capture at full strength fixes the shortage of incorrect predictions without reordering them. Both-pinned subtracts up to two from each branch, deep truncation, nearly every turn with either group live is asserted incorrect, and the metrics collapse, f1 0.267, the designed stress case behaving as designed.

**The slip attribution narrows.** The median drop falls to 0.003 while the top-KC drops persist at 0.2 to 0.55, the same concentrated set as F1, so the re-attribution is real for a small set of KCs and fragile as a median, an optimizer-sensitive summary rather than a broad effect.

## 8. Notes

- Every model keeps its fitted parameters at `.parameters` and the shared frozen chains at `.chains`, nothing here refits the chains.
- The chains' two columns arrive in GROUPS order, conceptual then procedural, and the strict compact-position check raises on any frame and chain-sequence disagreement rather than realigning silently.
- The winner's row name and betas get logged with the pick, and the F3 face reuses this notebook's shape with five per-family channels.